# テーブルと列を確認する
`smoke` と `local_experiment` の一覧を取得します。初期状態の Run All は一覧取得だけを行い、Qwen・preview・COUNT は実行しません。取得完了時刻は今回の読取が終わった時刻で、snapshot の作成時刻ではありません。
空の一覧、一部だけの一覧、取得失敗は別に表示します。各セルの再実行は新しい読取です。取消は Interrupt を1回使い、後片付けを待ってください。

In [ ]:
from IPython.display import HTML, display
from tools.local_experiment.notebook_lab.query import tables, describe, recover_metadata, render_metadata
from tools.local_experiment.notebook_lab.query import preview, count, render_preview, render_count
from tools.local_experiment.notebook.client import NotebookError

In [ ]:
inventories = {}
table_details = None
preview_result = None
count_result = None
for schema in ("smoke", "local_experiment"):
    observed = None
    try:
        observed = tables(schema)
    except NotebookError as error:
        print(schema, "の一覧を取得できませんでした。固定エラーコード:", str(error))
        break  # 失敗後の読取は自動で続けない
    else:
        inventories[schema] = observed
        display(HTML(render_metadata(observed)))

## テーブルを明示的に選ぶ
一覧にある `schema.table` を下の `selected_table` に設定して、このセルを実行します。列名・型・NULL 可否は取得時点の schema です。一覧に出ないテーブルを空とみなしたり、部分一覧を完全な一覧として扱ったりしません。

In [ ]:
selected_table = ""  # 例: smoke.fixture_v1
table_details = None
preview_result = None
count_result = None
listed_tables = {name for observed in inventories.values() for name in observed["metadata"]["tables"]}
if selected_table:
    if selected_table not in listed_tables:
        print("現在表示している一覧からテーブルを選んでください。必要なら一覧を再取得してください。")
    else:
        try:
            table_details = describe(selected_table)
        except NotebookError as error:
            print("列情報を取得できませんでした。固定エラーコード:", str(error))
        else:
            display(HTML(render_metadata(table_details)))

## 選択したテーブルで任意の実験を行う
以下は既存の自然文 preview と COUNT です。依頼を書き、対応する実行フラグを `True` にした場合だけ Qwen と Reader を呼びます。COUNT は全 snapshot を読む場合があります。列情報の取得に失敗した場合や選択を変えた場合は、先に上のセルで列情報を取得してください。

In [ ]:
run_preview = False
preview_request = "nameとidを1件表示"
run_count = False
count_request = "全行の件数を数えてください"
preview_result = None
count_result = None
described_table = None if table_details is None else table_details["metadata"]["schema"] + "." + table_details["metadata"]["table"]
if run_preview or run_count:
    if not selected_table or described_table != selected_table:
        print("選択したテーブルの列情報を先に取得してください。")
    else:
        try:
            if run_preview:
                preview_result = preview(selected_table, preview_request)
                display(HTML(render_preview(preview_result)))
            if run_count:
                count_result = count(selected_table, count_request)
                display(HTML(render_count(count_result)))
        except NotebookError as error:
            print("実験を完了できませんでした。固定エラーコード:", str(error))

## 保存したメタデータを再表示する
完了済み実行の token を指定します。取得当時の証跡を確認して表示し、追加のテーブル読取やモデル呼び出しは行いません。

In [ ]:
metadata_attempt = ""
saved_metadata = None
if metadata_attempt:
    try:
        saved_metadata = recover_metadata(metadata_attempt)
    except NotebookError as error:
        print("保存メタデータを確認できませんでした。固定エラーコード:", str(error))
    else:
        display(HTML(render_metadata(saved_metadata)))